In [1]:
# ! pip install pandas
# ! pip install regex
# ! pip install pymongo
# ! pip install dotenv
# ! pip install matplotlib

In [2]:
import pandas as pd
import sys
import os


In [3]:
import matplotlib
matplotlib.use("Agg")  # Non-interactive backend (no GUI)

import matplotlib.pyplot as plt
plt.ioff()  # Turn off interactive mode
plt.show = lambda *args, **kwargs: None 

In [4]:
bigcodebench = pd.read_csv("/Users/jin/Documents/GitHub/Code Reasoning Model Research Project/datasets/open_ended_format/bigcodebench_test.csv", header = 0, encoding='utf-8')
curr_dir = os.getcwd()
parent_dir = os.path.dirname(curr_dir)
proj_dir = os.path.dirname(parent_dir)
sys.path.append(proj_dir)

In [5]:
from code_generation.utility.humaneval_helper import CodeGenerationHumanEvalHelper
from code_generation.utility.bigcodebench_helper import CodeGenerationBigCodeBenchHelper
from database import MongoDBHelper

In [6]:
sample_qn = bigcodebench.iloc[9]
print(bigcodebench.columns.values)
prompt = sample_qn['complete_prompt']
canonical_solution = sample_qn['canonical_solution'].replace("\"", '"""').replace("\'", "'''")
instruct_prompt = sample_qn['instruct_prompt']
test = sample_qn['test'].replace("\"", '"""').replace("\'", "'''")
# original_test = sample_qn['test']
# print(prompt)

['task_id' 'complete_prompt' 'instruct_prompt' 'canonical_solution'
 'code_prompt' 'test' 'entry_point' 'doc_struct' 'libs']


In [7]:
prompt, qn_desc = CodeGenerationBigCodeBenchHelper.seperate_original_desciptions(prompt = sample_qn['complete_prompt'])

In [8]:
task_doc_string, example = CodeGenerationBigCodeBenchHelper.extract_examples(qn_desc)

In [9]:
natural_language_instruct, code_instruct= CodeGenerationBigCodeBenchHelper.split_code_from_instruct_prompt(instruct_prompt=instruct_prompt)

In [10]:
full_sol = CodeGenerationBigCodeBenchHelper.obtain_full_sol(
canonical_sol=canonical_solution, 
code_instruct=code_instruct,
test = test)

# Connecting to MongoDB

In [11]:
mongodbHelper = MongoDBHelper()
mongodbHelper.check_database_connectivity()

True

In [12]:
db = mongodbHelper.client["Base_Questions_DB"]
open_ended_db = db["BigCodeBench_Code_Generation"]

In [14]:
# %%script false --no-raise-error

to_check_example = []
to_check_test = []

try: 
    for idx in range(
        bigcodebench.__len__(),
        ):

        qn_id = f"BigCodeBencho{idx-len(to_check_example)-len(to_check_test)}"

        qn_details = open_ended_db.find_one({"_id" : qn_id})        # checking if this qn_id already exists in the db

        qn = bigcodebench.iloc[idx]

        original_prompt = qn['complete_prompt'].replace(r"\\x", r"\\\\x")
        canonical_solution = qn['canonical_solution']
        instruct_prompt = qn['instruct_prompt']

        test = qn['test']
        original_test_id = qn['task_id']

        prompt, qn_desc = CodeGenerationBigCodeBenchHelper.seperate_original_desciptions(original_prompt)

        task_doc_string, example = CodeGenerationBigCodeBenchHelper.extract_examples(qn_desc)

        natural_language_instruct, code_instruct= CodeGenerationBigCodeBenchHelper.split_code_from_instruct_prompt(instruct_prompt=instruct_prompt)
        try: 
            full_sol = CodeGenerationBigCodeBenchHelper.obtain_full_sol(
                canonical_sol=canonical_solution, 
                code_instruct=code_instruct,
                test = test)
        except Exception as e:
            print(e)
            print(original_test_id)
            to_check_test.append(original_test_id)
            continue
        
        entry_dict = {
            "_id" : qn_id,
            "qn" : code_instruct,
            "task_doc_string": task_doc_string,
            "canon_solution" : canonical_solution,
            "qn_desc" : qn_desc,
            "examples": example,
            "check" : test,
            "original_id": original_test_id
        }
        if qn_details is None:
            open_ended_db.insert_one(entry_dict)
            print('Added entry to database: {id}'.format(id = qn_id))
        else:
            open_ended_db.update_one({"_id" : qn_id}, update = {"$set": entry_dict})
            print('Updated existing entry in database: {id}'.format(id = qn_id))
except KeyboardInterrupt:
    print(original_test_id)
except Exception as e:
    print(original_test_id)
    print(e)
    print(to_check_test)


print(to_check_example if len(to_check_example) > 0 else "All cases contains examples. Nothing to check!")
print(to_check_test if len(to_check_test) > 0 else "All test cases passed. Nothing to check!")

Added entry to database: BigCodeBencho0
Added entry to database: BigCodeBencho1
Added entry to database: BigCodeBencho2
Added entry to database: BigCodeBencho3
Added entry to database: BigCodeBencho4
Added entry to database: BigCodeBencho5
Added entry to database: BigCodeBencho6
Added entry to database: BigCodeBencho7
Added entry to database: BigCodeBencho8
Added entry to database: BigCodeBencho9
Added entry to database: BigCodeBencho10
Added entry to database: BigCodeBencho11
Added entry to database: BigCodeBencho12
Added entry to database: BigCodeBencho13
Added entry to database: BigCodeBencho14
Added entry to database: BigCodeBencho15


tar: Removing leading '/' from member names
a var/folders/gt/q99q3ns57l75nmgnyb_khwnm0000gn/T/tmp6pth59u6/file_4.log
a var/folders/gt/q99q3ns57l75nmgnyb_khwnm0000gn/T/tmp6pth59u6/file_0.log
a var/folders/gt/q99q3ns57l75nmgnyb_khwnm0000gn/T/tmp6pth59u6/file_1.log
a var/folders/gt/q99q3ns57l75nmgnyb_khwnm0000gn/T/tmp6pth59u6/file_3.log
a var/folders/gt/q99q3ns57l75nmgnyb_khwnm0000gn/T/tmp6pth59u6/file_2.log
tar: Removing leading '/' from member names
a var/folders/gt/q99q3ns57l75nmgnyb_khwnm0000gn/T/tmp22h8pbyp/file_4.log
a var/folders/gt/q99q3ns57l75nmgnyb_khwnm0000gn/T/tmp22h8pbyp/file_0.log
a var/folders/gt/q99q3ns57l75nmgnyb_khwnm0000gn/T/tmp22h8pbyp/file_1.log
a var/folders/gt/q99q3ns57l75nmgnyb_khwnm0000gn/T/tmp22h8pbyp/file_3.log
a var/folders/gt/q99q3ns57l75nmgnyb_khwnm0000gn/T/tmp22h8pbyp/file_2.log


Added entry to database: BigCodeBencho16
Added entry to database: BigCodeBencho17
Added entry to database: BigCodeBencho18
Added entry to database: BigCodeBencho19
Added entry to database: BigCodeBencho20
Added entry to database: BigCodeBencho21
Added entry to database: BigCodeBencho22
Added entry to database: BigCodeBencho23
Added entry to database: BigCodeBencho24
Added entry to database: BigCodeBencho25
Added entry to database: BigCodeBencho26
Added entry to database: BigCodeBencho27
Added entry to database: BigCodeBencho28
Added entry to database: BigCodeBencho29
Added entry to database: BigCodeBencho30
Added entry to database: BigCodeBencho31
Added entry to database: BigCodeBencho32
Added entry to database: BigCodeBencho33
Added entry to database: BigCodeBencho34
Added entry to database: BigCodeBencho35
Added entry to database: BigCodeBencho36
Added entry to database: BigCodeBencho37
Added entry to database: BigCodeBencho38
Added entry to database: BigCodeBencho39
Added entry to d

[nltk_data] Downloading package punkt to /Users/jin/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


Added entry to database: BigCodeBencho173
Added entry to database: BigCodeBencho174
Added entry to database: BigCodeBencho175
Added entry to database: BigCodeBencho176
Added entry to database: BigCodeBencho177
Added entry to database: BigCodeBencho178
Added entry to database: BigCodeBencho179
Added entry to database: BigCodeBencho180
Added entry to database: BigCodeBencho181
Added entry to database: BigCodeBencho182
Added entry to database: BigCodeBencho183
Added entry to database: BigCodeBencho184
Added entry to database: BigCodeBencho185
Added entry to database: BigCodeBencho186
Added entry to database: BigCodeBencho187
Added entry to database: BigCodeBencho188
Added entry to database: BigCodeBencho189
Added entry to database: BigCodeBencho190
Added entry to database: BigCodeBencho191
Added entry to database: BigCodeBencho192
Added entry to database: BigCodeBencho193
Added entry to database: BigCodeBencho194
Added entry to database: BigCodeBencho195
Added entry to database: BigCodeBe

...

Added entry to database: BigCodeBencho705
Added entry to database: BigCodeBencho706
Added entry to database: BigCodeBencho707
Added entry to database: BigCodeBencho708
Added entry to database: BigCodeBencho709
Added entry to database: BigCodeBencho710
Added entry to database: BigCodeBencho711
Added entry to database: BigCodeBencho712


[nltk_data] Downloading package punkt to /Users/jin/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/jin/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


Full solution failed due to following errors from test case: [(<builtins.TestCases testMethod=test_case_1>, 'Traceback (most recent call last):\n  File "<string>", line 8, in test_case_1\n  File "<string>", line 8, in task_func\n  File "/Users/jin/Documents/GitHub/Code Reasoning Model Research Project/bigcodebenchvenv/lib/python3.11/site-packages/nltk/tag/__init__.py", line 168, in pos_tag\n    tagger = _get_tagger(lang)\n             ^^^^^^^^^^^^^^^^^\n  File "/Users/jin/Documents/GitHub/Code Reasoning Model Research Project/bigcodebenchvenv/lib/python3.11/site-packages/nltk/tag/__init__.py", line 110, in _get_tagger\n    tagger = PerceptronTagger()\n             ^^^^^^^^^^^^^^^^^^\n  File "/Users/jin/Documents/GitHub/Code Reasoning Model Research Project/bigcodebenchvenv/lib/python3.11/site-packages/nltk/tag/perceptron.py", line 183, in __init__\n    self.load_from_json(lang)\n  File "/Users/jin/Documents/GitHub/Code Reasoning Model Research Project/bigcodebenchvenv/lib/python3.11/si

[nltk_data] Downloading package stopwords to /Users/jin/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package stopwords to /Users/jin/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Added entry to database: BigCodeBencho785
Added entry to database: BigCodeBencho786
Added entry to database: BigCodeBencho787
Added entry to database: BigCodeBencho788
Added entry to database: BigCodeBencho789
Added entry to database: BigCodeBencho790
Added entry to database: BigCodeBencho791
Added entry to database: BigCodeBencho792
Added entry to database: BigCodeBencho793
Added entry to database: BigCodeBencho794
Added entry to database: BigCodeBencho795
Added entry to database: BigCodeBencho796
Added entry to database: BigCodeBencho797
Added entry to database: BigCodeBencho798
Added entry to database: BigCodeBencho799
Added entry to database: BigCodeBencho800
Added entry to database: BigCodeBencho801
Added entry to database: BigCodeBencho802
Added entry to database: BigCodeBencho803
Added entry to database: BigCodeBencho804
Added entry to database: BigCodeBencho805
Added entry to database: BigCodeBencho806
Added entry to database: BigCodeBencho807
Added entry to database: BigCodeBe

In [ ]:
%%script false --no-raise-error
samples = {'BigCodeBench/871', 'BigCodeBench/940', 'BigCodeBench/362', 'BigCodeBench/1022', 'BigCodeBench/779', 'BigCodeBench/1049', 'BigCodeBench/83', 'BigCodeBench/348', 'BigCodeBench/131', 'BigCodeBench/724', 'BigCodeBench/80', 'BigCodeBench/372', 'BigCodeBench/158', 'BigCodeBench/498', 'BigCodeBench/723', 'BigCodeBench/1084', 'BigCodeBench/1129', 'BigCodeBench/808', 'BigCodeBench/82', 'BigCodeBench/656', 'BigCodeBench/632', 'BigCodeBench/806', 'BigCodeBench/274', 'BigCodeBench/130', 'BigCodeBench/227', 'BigCodeBench/245', 'BigCodeBench/346', 'BigCodeBench/847', 'BigCodeBench/596', 'BigCodeBench/370', 'BigCodeBench/495', 'BigCodeBench/501', 'BigCodeBench/972', 'BigCodeBench/634', 'BigCodeBench/1109', 'BigCodeBench/987', 'BigCodeBench/202', 'BigCodeBench/358', 'BigCodeBench/205', 'BigCodeBench/964', 'BigCodeBench/683', 'BigCodeBench/270', 'BigCodeBench/926', 'BigCodeBench/490', 'BigCodeBench/565', 'BigCodeBench/203', 'BigCodeBench/985', 'BigCodeBench/39', 'BigCodeBench/655', 'BigCodeBench/924', 'BigCodeBench/708', 'BigCodeBench/360', 'BigCodeBench/374', 'BigCodeBench/81', 'BigCodeBench/593', 'BigCodeBench/746', 'BigCodeBench/383', 'BigCodeBench/726', 'BigCodeBench/728', 'BigCodeBench/215', 'BigCodeBench/101', 'BigCodeBench/334', 'BigCodeBench/734', 'BigCodeBench/629', 'BigCodeBench/461', 'BigCodeBench/115', 'BigCodeBench/132', 'BigCodeBench/812', 'BigCodeBench/1020', 'BigCodeBench/364', 'BigCodeBench/361', 'BigCodeBench/177', 'BigCodeBench/657', 'BigCodeBench/927', 'BigCodeBench/220', 'BigCodeBench/736', 'BigCodeBench/1028', 'BigCodeBench/658', 'BigCodeBench/363', 'BigCodeBench/804', 'BigCodeBench/986', 'BigCodeBench/1009', 'BigCodeBench/686', 'BigCodeBench/1008', 'BigCodeBench/761', 'BigCodeBench/1095', 'BigCodeBench/1124', 'BigCodeBench/458', 'BigCodeBench/412', 'BigCodeBench/844', 'BigCodeBench/612', 'BigCodeBench/849', 'BigCodeBench/192', 'BigCodeBench/994', 'BigCodeBench/1000', 'BigCodeBench/577', 'BigCodeBench/867'}

to_check_example = []
to_check_test = set()

for qn_id in samples:
    try: 

        print(qn_id)
        idx = int(qn_id.split('BigCodeBench/')[-1])

        qn_id = f"BigCodeBencho{idx-len(to_check_example)-len(samples)}"

        qn_details = open_ended_db.find_one({"_id" : qn_id})        # checking if this qn_id already exists in the db

        qn = bigcodebench.iloc[idx]
        original_prompt = qn['complete_prompt']
        canonical_solution = qn['canonical_solution'].replace("\"", '"""').replace("\'", "'''")
        instruct_prompt = qn['instruct_prompt']
        test = qn['test'].replace("\"", '"""').replace("\'", "'''")
        original_test_id = qn['task_id']

        prompt, qn_desc = CodeGenerationBigCodeBenchHelper.seperate_original_desciptions(original_prompt)

        task_doc_string, example = CodeGenerationBigCodeBenchHelper.extract_examples(qn_desc)

        natural_language_instruct, code_instruct= CodeGenerationBigCodeBenchHelper.split_code_from_instruct_prompt(instruct_prompt=instruct_prompt)
        try: 
            full_sol = CodeGenerationBigCodeBenchHelper.obtain_full_sol(
                canonical_sol=canonical_solution, 
                code_instruct=code_instruct,
                test = test)
        except Exception as e:
            print(e)
            to_check_test.add(original_test_id)
            continue
        
        entry_dict = {
            "_id" : qn_id,
            "qn" : code_instruct,
            "task_doc_string": task_doc_string,
            "canon_solution" : canonical_solution,
            "qn_desc" : qn_desc,
            "example": example,
            "check" : test,
            "original_id": original_test_id
        }
        if qn_details is None:
            open_ended_db.insert_one(entry_dict)
            print('Added entry to database: {id}'.format(id = qn_id))
        else:
            open_ended_db.update_one({"_id" : qn_id}, update = {"$set": entry_dict})
            print('Updated existing entry in database: {id}'.format(id = qn_id))
    except KeyboardInterrupt:
        print(original_test_id)
    except Exception as e:
        print(original_test_id)
        print(e)
        continue


print(to_check_example if len(to_check_example) > 0 else "All cases contains examples. Nothing to check!")
print(to_check_test if len(to_check_test) > 0 else "All test cases passed. Nothing to check!")

BigCodeBench/926
Added entry to database: BigCodeBencho829
BigCodeBench/565
Added entry to database: BigCodeBencho468
BigCodeBench/220
Module '_tkinter' not found. Installing...
Command '['/Users/jin/Documents/GitHub/Code Reasoning Model Research Project/bigcodebenchvenv/bin/python', '-m', 'pip', 'install', '_tkinter']' returned non-zero exit status 1.
BigCodeBench/728
Added entry to database: BigCodeBencho631
BigCodeBench/372
Module 'exceptions' not found. Installing...
Command '['/Users/jin/Documents/GitHub/Code Reasoning Model Research Project/bigcodebenchvenv/bin/python', '-m', 'pip', 'install', 'exceptions']' returned non-zero exit status 1.
BigCodeBench/412
Full solution failed due to following errors from test case: [(<builtins.TestCases testMethod=test_decode_base64>, 'Traceback (most recent call last):\n  File "<string>", line 12, in test_decode_base64\n  File "<string>", line 8, in task_func\n  File "/Users/jin/.pyenv/versions/3.11.0/lib/python3.11/json/__init__.py", line 293

[nltk_data] Downloading package stopwords to /Users/jin/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Added entry to database: BigCodeBencho558
BigCodeBench/101
Full solution failed due to following errors from test case: [(<builtins.TestCases testMethod=test_heatmap_values>, 'Traceback (most recent call last):\n  File "<string>", line 17, in test_heatmap_values\n  File "/Users/jin/.pyenv/versions/3.11.0/lib/python3.11/unittest/case.py", line 904, in assertAlmostEqual\n    diff = abs(first - second)\n               ~~~~~~^~~~~~~~\nTypeError: unsupported operand type(s) for -: \'list\' and \'list\'\n')]
BigCodeBench/80
Full solution failed due to following errors from test case: [(<builtins.TestCases testMethod=test_app_creation>, 'Traceback (most recent call last):\n  File "<string>", line 19, in test_app_creation\n  File "<string>", line 7, in task_func\n  File "/Users/jin/Documents/GitHub/Code Reasoning Model Research Project/bigcodebenchvenv/lib/python3.11/site-packages/flask/app.py", line 239, in __init__\n    super().__init__(\n  File "/Users/jin/Documents/GitHub/Code Reasoning Mo

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /Users/jin/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!
[nltk_data] Downloading package punkt to /Users/jin/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


Full solution failed due to following errors from test case: [(<builtins.TestCases testMethod=test_case_sensitive_handling>, 'Traceback (most recent call last):\n  File "<string>", line 65, in test_case_sensitive_handling\n  File "<string>", line 16, in task_func\n  File "/Users/jin/Documents/GitHub/Code Reasoning Model Research Project/bigcodebenchvenv/lib/python3.11/site-packages/nltk/tokenize/__init__.py", line 142, in word_tokenize\n    sentences = [text] if preserve_line else sent_tokenize(text, language)\n                                             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File "/Users/jin/Documents/GitHub/Code Reasoning Model Research Project/bigcodebenchvenv/lib/python3.11/site-packages/nltk/tokenize/__init__.py", line 119, in sent_tokenize\n    tokenizer = _get_punkt_tokenizer(language)\n                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File "/Users/jin/Documents/GitHub/Code Reasoning Model Research Project/bigcodebenchvenv/lib/python3.11/site-packages/nltk/tokenize/_

...

Full solution failed due to following errors from test case: [(<builtins.TestCases testMethod=test_custom_sheet_name>, 'Traceback (most recent call last):\n  File "<string>", line 10, in task_func\n  File "/Users/jin/Documents/GitHub/Code Reasoning Model Research Project/bigcodebenchvenv/lib/python3.11/site-packages/pandas/io/json/_json.py", line 815, in read_json\n    return json_reader.read()\n           ^^^^^^^^^^^^^^^^^^\n  File "/Users/jin/Documents/GitHub/Code Reasoning Model Research Project/bigcodebenchvenv/lib/python3.11/site-packages/pandas/io/json/_json.py", line 1014, in read\n    obj = self._get_object_parser(self.data)\n          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File "/Users/jin/Documents/GitHub/Code Reasoning Model Research Project/bigcodebenchvenv/lib/python3.11/site-packages/pandas/io/json/_json.py", line 1040, in _get_object_parser\n    obj = FrameParser(json, **kwargs).parse()\n          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File "/Users/jin/Documents/GitHub/Co

[nltk_data] Downloading package punkt to /Users/jin/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/jin/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


Added entry to database: BigCodeBencho33
BigCodeBench/81
Full solution failed due to following errors from test case: [(<builtins.TestCases testMethod=test_api_endpoint_configuration>, 'Traceback (most recent call last):\n  File "<string>", line 15, in test_api_endpoint_configuration\n  File "<string>", line 5, in task_func\n  File "/Users/jin/Documents/GitHub/Code Reasoning Model Research Project/bigcodebenchvenv/lib/python3.11/site-packages/flask/app.py", line 239, in __init__\n    super().__init__(\n  File "/Users/jin/Documents/GitHub/Code Reasoning Model Research Project/bigcodebenchvenv/lib/python3.11/site-packages/flask/sansio/app.py", line 295, in __init__\n    super().__init__(\n  File "/Users/jin/Documents/GitHub/Code Reasoning Model Research Project/bigcodebenchvenv/lib/python3.11/site-packages/flask/sansio/scaffold.py", line 96, in __init__\n    root_path = get_root_path(self.import_name)\n                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File "/Users/jin/Documents/GitHub/C

[nltk_data] Downloading package stopwords to /Users/jin/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Full solution failed due to following errors from test case: [(<builtins.TestCases testMethod=test_large_numbers>, 'Traceback (most recent call last):\n  File "<string>", line 22, in test_large_numbers\n  File "<string>", line 11, in task_func\n  File "/Users/jin/.pyenv/versions/3.11.0/lib/python3.11/multiprocessing/pool.py", line 375, in starmap\n    return self._map_async(func, iterable, starmapstar, chunksize).get()\n           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File "/Users/jin/.pyenv/versions/3.11.0/lib/python3.11/multiprocessing/pool.py", line 774, in get\n    raise self._value\n  File "/Users/jin/.pyenv/versions/3.11.0/lib/python3.11/multiprocessing/pool.py", line 540, in _handle_tasks\n    put(task)\n  File "/Users/jin/.pyenv/versions/3.11.0/lib/python3.11/multiprocessing/connection.py", line 205, in send\n    self._send_bytes(_ForkingPickler.dumps(obj))\n                     ^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File "/Users/jin/.pyenv/versions/3.11.0/lib

In [ ]:
p = {'BigCodeBench/200', 'BigCodeBench/960', 'BigCodeBench/186', 'BigCodeBench/484', 'BigCodeBench/579', 'BigCodeBench/o37', 'BigCodeBench/652', 'BigCodeBench/764', 'BigCodeBench/900', 'BigCodeBench/929', 'BigCodeBench/342', 'BigCodeBench/448', 'BigCodeBench/794', 'BigCodeBench/423', 'BigCodeBench/320', 'BigCodeBench/997', 'BigCodeBench/888', 'BigCodeBench/669', 'BigCodeBench/774', 'BigCodeBench/616', 'BigCodeBench/853', 'BigCodeBench/989', 'BigCodeBench/956', 'BigCodeBench/156', 'BigCodeBench/309', 'BigCodeBench/162', 'BigCodeBench/356', 'BigCodeBench/847', 'BigCodeBench/o75', 'BigCodeBench/659', 'BigCodeBench/308', 'BigCodeBench/248', 'BigCodeBench/881', 'BigCodeBench/o28', 'BigCodeBench/249', 'BigCodeBench/703', 'BigCodeBench/459', 'BigCodeBench/773', 'BigCodeBench/341', 'BigCodeBench/532', 'BigCodeBench/627', 'BigCodeBench/016', 'BigCodeBench/734', 'BigCodeBench/334', 'BigCodeBench/846', 'BigCodeBench/657', 'BigCodeBench/996', 'BigCodeBench/949', 'BigCodeBench/256', 'BigCodeBench/938', 'BigCodeBench/148', 'BigCodeBench/o89', 'BigCodeBench/185', 'BigCodeBench/553', 'BigCodeBench/561', 'BigCodeBench/o15', 'BigCodeBench/670', 'BigCodeBench/030', 'BigCodeBench/119', 'BigCodeBench/o50', 'BigCodeBench/874', 'BigCodeBench/o93', 'BigCodeBench/912', 'BigCodeBench/590', 'BigCodeBench/224', 'BigCodeBench/580', 'BigCodeBench/104', 'BigCodeBench/531', 'BigCodeBench/o76', 'BigCodeBench/545', 'BigCodeBench/547', 'BigCodeBench/o52', 'BigCodeBench/422', 'BigCodeBench/947', 'BigCodeBench/910', 'BigCodeBench/520', 'BigCodeBench/615', 'BigCodeBench/710', 'BigCodeBench/648', 'BigCodeBench/930', 'BigCodeBench/859', 'BigCodeBench/175', 'BigCodeBench/105', 'BigCodeBench/771', 'BigCodeBench/120', 'BigCodeBench/455', 'BigCodeBench/340', 'BigCodeBench/458', 'BigCodeBench/174', 'BigCodeBench/008', 'BigCodeBench/350', 'BigCodeBench/601', 'BigCodeBench/577', 'BigCodeBench/005', 'BigCodeBench/701', 'BigCodeBench/466', 'BigCodeBench/004', 'BigCodeBench/021', 'BigCodeBench/775', 'BigCodeBench/741', 'BigCodeBench/664', 'BigCodeBench/250', 'BigCodeBench/995', 'BigCodeBench/398', 'BigCodeBench/369'}
print(len(p))

x = ['BigCodeBench/80', 'BigCodeBench/81', 'BigCodeBench/82', 'BigCodeBench/83', 'BigCodeBench/101', 'BigCodeBench/115', 'BigCodeBench/177', 'BigCodeBench/205', 'BigCodeBench/220', 'BigCodeBench/245', 'BigCodeBench/334', 'BigCodeBench/363', 'BigCodeBench/372', 'BigCodeBench/383', 'BigCodeBench/501', 'BigCodeBench/593', 'BigCodeBench/596', 'BigCodeBench/612', 'BigCodeBench/634', 'BigCodeBench/683', 'BigCodeBench/686', 'BigCodeBench/734', 'BigCodeBench/736', 'BigCodeBench/779', 'BigCodeBench/940', 'BigCodeBench/964', 'BigCodeBench/1028', 'BigCodeBench/1109']
print(len(x))

c1 = 0
for i in p:
    if i not in x:
        print(f"Failed in p: {i}")
        c1+= 1
c2 = 0
for i in x:
    if i not in p:
        print(f'failed in x: {i}')
        c2 += 1

print(c1, c2)

105
28
Failed in p: BigCodeBench/669
Failed in p: BigCodeBench/o15
Failed in p: BigCodeBench/601
Failed in p: BigCodeBench/249
Failed in p: BigCodeBench/664
Failed in p: BigCodeBench/616
Failed in p: BigCodeBench/466
Failed in p: BigCodeBench/o28
Failed in p: BigCodeBench/764
Failed in p: BigCodeBench/881
Failed in p: BigCodeBench/008
Failed in p: BigCodeBench/947
Failed in p: BigCodeBench/995
Failed in p: BigCodeBench/561
Failed in p: BigCodeBench/174
Failed in p: BigCodeBench/888
Failed in p: BigCodeBench/021
Failed in p: BigCodeBench/847
Failed in p: BigCodeBench/938
Failed in p: BigCodeBench/648
Failed in p: BigCodeBench/200
Failed in p: BigCodeBench/794
Failed in p: BigCodeBench/250
Failed in p: BigCodeBench/553
Failed in p: BigCodeBench/741
Failed in p: BigCodeBench/o75
Failed in p: BigCodeBench/652
Failed in p: BigCodeBench/369
Failed in p: BigCodeBench/771
Failed in p: BigCodeBench/930
Failed in p: BigCodeBench/448
Failed in p: BigCodeBench/846
Failed in p: BigCodeBench/342
Fai